<a href="https://colab.research.google.com/github/av-jones/DimABSA/blob/main/AJ_PhD_Week2_RoBERTa_TEST_SET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PhD Week 2: RoBERTa Fine-Tuning — TEST SET EVALUATION

**Changes from dev version:**
- Trains on **full training set** (train + valid combined)
- Evaluates on **official test set** with gold labels
- Reports per-domain and overall **test RMSE**


In [ ]:
!pip install -q transformers datasets accelerate
from google.colab import drive
#drive.mount('/content/drive')
import json, torch, numpy as np, re
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from datasets import Dataset
from torch import nn
from tqdm import tqdm

In [ ]:
def load_jsonl(fp):
    return [json.loads(l) for l in open(fp) if l.strip()]

# ── FULL training data (restaurant + laptop) ──────────────────────────────────
train_rest   = load_jsonl('/content/sample_data/eng_restaurant_train_alltasks.jsonl')
train_laptop = load_jsonl('/content/sample_data/eng_laptop_train_alltasks.jsonl')

# Also load original dev split to combine into full training set
dev_rest     = load_jsonl('/content/sample_data/eng_restaurant_valid.jsonl')
dev_laptop   = load_jsonl('/content/sample_data/eng_laptop_valid.jsonl')

# ── Test set: task1 file (aspects) + gold labels ──────────────────────────────
test_rest    = load_jsonl('/content/sample_data/eng_restaurant_test_task1.jsonl')
test_laptop  = load_jsonl('/content/sample_data/eng_laptop_test_task1.jsonl')
gold_rest    = load_jsonl('/content/sample_data/eng_restaurant_test_gold.jsonl')
gold_laptop  = load_jsonl('/content/sample_data/eng_laptop_test_gold.jsonl')

def extract_pairs(data, domain):
    pairs = []
    for item in data:
        for quad in item['Quadruplet']:
            aspect = quad['Aspect'] if quad['Aspect'] != 'NULL' else quad['Category']
            v, a = map(float, quad['VA'].split('#'))
            pairs.append({'text': f"{item['Text']} [SEP] {aspect}", 'valence': v, 'arousal': a})
    return pairs

def extract_pairs_from_gold(data, domain):
    """Extract pairs from gold test file (uses Aspect_VA format)"""
    pairs = []
    for item in data:
        for av in item['Aspect_VA']:
            v, a = map(float, av['VA'].split('#'))
            pairs.append({'text': f"{item['Text']} [SEP] {av['Aspect']}", 'valence': v, 'arousal': a})
    return pairs

# Combine full train + dev for training
train_pairs = (extract_pairs(train_rest, 'rest') + extract_pairs(train_laptop, 'laptop') +
               extract_pairs(dev_rest, 'rest')   + extract_pairs(dev_laptop, 'laptop'))

# Test pairs from gold labels (for RMSE evaluation)
test_rest_pairs   = extract_pairs_from_gold(gold_rest,   'rest')
test_laptop_pairs = extract_pairs_from_gold(gold_laptop, 'laptop')

print(f'Full train: {len(train_pairs)}')
print(f'Test rest:  {len(test_rest_pairs)}')
print(f'Test laptop:{len(test_laptop_pairs)}')


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    'text':    [p['text']    for p in train_pairs],
    'valence': [p['valence'] for p in train_pairs],
    'arousal': [p['arousal'] for p in train_pairs]
})

# No validation dataset — we train on everything, evaluate on test
print(f'Train dataset: {len(train_dataset)} samples')


In [ ]:
MODEL_NAME = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class RoBERTaVA(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.regressor = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(self.roberta.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 2)
        )
    def forward(self, input_ids, attention_mask):
        return self.regressor(self.roberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :])

model = RoBERTaVA(MODEL_NAME)
print('✓ RoBERTa model created')

In [ ]:
def tokenize(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

train_tok = train_dataset.map(tokenize, batched=True)
train_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'valence', 'arousal'])
print('✓ Tokenization complete')


In [ ]:
from transformers import Trainer, TrainingArguments
import torch.nn as nn

class VATrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=0):
        labels = torch.stack([inputs.pop('valence').float(), inputs.pop('arousal').float()], dim=1)
        outputs = model(**inputs)
        loss = nn.MSELoss()(outputs, labels)
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir='./roberta_va',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy='no',      # No validation set — train on everything
    save_strategy='no',
    report_to='none',
    label_names=['valence', 'arousal']
)

trainer = VATrainer(model=model, args=args, train_dataset=train_tok)
print('Starting training on full dataset (~20-25 min)...')
trainer.train()
print('✓ Training complete')


In [ ]:
def predict_va(text, aspect):
    inputs = tokenizer(f'{text} [SEP] {aspect}', return_tensors='pt', truncation=True, max_length=128)
    inputs = {k: v.to(model.roberta.device) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    v, a = outputs[0].cpu().numpy()
    return max(1.0, min(9.0, float(v))), max(1.0, min(9.0, float(a)))

def compute_rmse(pairs, label):
    errors = []
    model.eval()
    for p in tqdm(pairs, desc=f'  RMSE {label}', leave=False):
        parts  = p['text'].split(' [SEP] ')
        text   = parts[0]
        aspect = parts[1] if len(parts) > 1 else ''
        pred_v, pred_a = predict_va(text, aspect)
        errors.append((pred_v - p['valence'])**2 + (pred_a - p['arousal'])**2)
    return round(float(np.sqrt(np.mean(errors))), 4) if errors else None

print('Evaluating on TEST set...')
rmse_rest_test   = compute_rmse(test_rest_pairs,   'Restaurant TEST')
rmse_laptop_test = compute_rmse(test_laptop_pairs, 'Laptop TEST')

n_rest, n_laptop = len(test_rest_pairs), len(test_laptop_pairs)
rmse_overall_test = round(float(np.sqrt(
    (n_rest * rmse_rest_test**2 + n_laptop * rmse_laptop_test**2) / (n_rest + n_laptop)
)), 4)

print(f'\n' + '='*50)
print(f'TEST RMSE  Restaurant : {rmse_rest_test}')
print(f'TEST RMSE  Laptop     : {rmse_laptop_test}')
print(f'TEST RMSE  Overall    : {rmse_overall_test}')
print(f'Target: < 1.0')
print('='*50)


In [ ]:
# Generate predictions in submission format (for ensemble / future use)
def predict_test(test_data, label):
    preds = []
    for item in tqdm(test_data, desc=f'  Predicting {label}'):
        av_list = []
        for aspect in item['Aspect']:
            v, a = predict_va(item['Text'], aspect)
            av_list.append({'Aspect': aspect, 'VA': f'{v:.2f}#{a:.2f}'})
        preds.append({'ID': item['ID'], 'Aspect_VA': av_list})
    return preds

def save_jsonl(preds, path):
    with open(path, 'w') as f:
        for p in preds: f.write(json.dumps(p) + '\n')

print('Generating submission predictions...')
rest_preds   = predict_test(test_rest,   'Restaurant')
laptop_preds = predict_test(test_laptop, 'Laptop')

save_jsonl(rest_preds,   '/content/sample_data/pred_eng_restaurant_TEST.jsonl')
save_jsonl(laptop_preds, '/content/sample_data/pred_eng_laptop_TEST.jsonl')

from google.colab import files
files.download('/content/sample_data/pred_eng_restaurant_TEST.jsonl')
files.download('/content/sample_data/pred_eng_laptop_TEST.jsonl')
print('✓ Predictions saved and downloaded')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 📋 EXPERIMENT LOGGER — PhD Week 2: RoBERTa WITHOUT RDoC Signals
#
# REPLACES Cell 8 in your notebook. Run after training completes.
#
#   ✅ Generates predictions for restaurant & laptop test sets
#   ✅ Saves predictions to a timestamped folder in Google Drive
#   ✅ Computes RMSE separately for restaurant & laptop on full val set
#   ✅ Pulls train loss + val loss per epoch from trainer.state
#   ✅ Logs total training runtime
#   ✅ Writes Drive file paths into the Linked Artifact column
#   ✅ Logs everything to your Master Excel sheet
#
# ✏️  Only edit the CONFIG block — nothing else needs changing run to run.
# ═══════════════════════════════════════════════════════════════════════════════

import openpyxl, datetime, os, json, numpy as np
from tqdm import tqdm
from openpyxl.styles import Font

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CONFIG                                                                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
EXCEL_PATH  = "/content/drive/MyDrive/ML_Experiments/ML_NLP_Experiment_Log.xlsx"
DRIVE_PREDS = "/content/drive/MyDrive/ML_Experiments/Predictions"
SHEET_NAME  = "Experiment Log"
FIRST_ROW   = 4
EXP_NAME    = "PhD_Wk2_RoBERTa_NO_RDoC_TEST"
RESEARCHER  = "avjones"
NEXT_STEPS  = "Trained on full data, evaluated on test set"
# ══════════════════════════════════════════════════════════════════════════════


# ── GUARD: make sure Drive is accessible ──────────────────────────────────────
if not os.path.exists(EXCEL_PATH):
    raise FileNotFoundError(
        f"\n❌  Excel log not found at:\n    {EXCEL_PATH}\n"
        "    → Check EXCEL_PATH above and confirm Drive is mounted (Cell 1)."
    )


# ── STEP 1: TOTAL TRAINING RUNTIME ────────────────────────────────────────────
try:
    runtime_entry = next(
        (e for e in reversed(trainer.state.log_history) if 'train_runtime' in e), None
    )
    if runtime_entry:
        total_secs    = runtime_entry['train_runtime']
        mins, secs    = int(total_secs // 60), int(total_secs % 60)
        runtime_str   = f"{mins}m {secs}s"
        samples_per_s = round(runtime_entry.get('train_samples_per_second', 0), 2)
    else:
        runtime_str, samples_per_s = "N/A", "?"
except Exception as e:
    runtime_str, samples_per_s = f"Error: {e}", "?"

print(f"⏱️   Training time  : {runtime_str}  ({samples_per_s} samples/sec)")


# ── STEP 2: TRAIN LOSS & VAL LOSS PER EPOCH ───────────────────────────────────
try:
    log_history  = trainer.state.log_history

    # Each epoch logs a 'loss' entry (train) and an 'eval_loss' entry (val)
    train_by_epoch = {int(e['epoch']): round(e['loss'], 4)
                      for e in log_history if 'loss' in e and 'eval_loss' not in e}
    val_by_epoch   = {int(e['epoch']): round(e['eval_loss'], 4)
                      for e in log_history if 'eval_loss' in e}

    train_loss_str = "  ".join([f"E{ep}:{loss}" for ep, loss in sorted(train_by_epoch.items())])
    val_loss_str   = "  ".join([f"E{ep}:{loss}" for ep, loss in sorted(val_by_epoch.items())])

    final_train_loss = list(train_by_epoch.values())[-1] if train_by_epoch else None
    final_val_loss   = list(val_by_epoch.values())[-1]   if val_by_epoch   else None
    best_ckpt        = os.path.basename(trainer.state.best_model_checkpoint or "N/A")

    print(f"📉  Train loss/epoch: {train_loss_str}")
    print(f"📉  Val   loss/epoch: {val_loss_str}")
    print(f"🏆  Best checkpoint : {best_ckpt}")

except Exception as e:
    train_loss_str = val_loss_str = f"Error: {e}"
    final_train_loss = final_val_loss = None
    best_ckpt = "N/A"


# ── STEP 3: RMSE PER DOMAIN ON FULL VALIDATION SET ────────────────────────────
def compute_rmse(pairs, label):
    errors = []
    model.eval()
    for p in tqdm(pairs, desc=f"  RMSE {label}", leave=False):
        parts  = p['text'].split(' [SEP] ')
        text   = parts[0]
        aspect = parts[1] if len(parts) > 1 else ''
        pred_v, pred_a = predict_va(text, aspect)
        errors.append((pred_v - p['valence'])**2 + (pred_a - p['arousal'])**2)
    return round(float(np.sqrt(np.mean(errors))), 4) if errors else None

print("\n⏳  Reading TEST RMSE (already computed above)...")
# rmse_rest_test, rmse_laptop_test, rmse_overall_test already computed in Cell 7
print(f"\n  ✅  Restaurant  TEST RMSE : {rmse_rest_test}")
print(f"  ✅  Laptop      TEST RMSE : {rmse_laptop_test}")
print(f"  ✅  Overall     TEST RMSE : {rmse_overall_test}")


# ── STEP 4: AUTO-GENERATE EXP ID + RUN # ──────────────────────────────────────
wb = openpyxl.load_workbook(EXCEL_PATH)
ws = wb[SHEET_NAME]

max_id = 0
for row in ws.iter_rows(min_row=FIRST_ROW, max_col=1, values_only=True):
    v = row[0]
    if v and isinstance(v, str) and v.startswith("EXP-"):
        try: max_id = max(max_id, int(v.split("-")[1]))
        except ValueError: pass
exp_id = f"EXP-{max_id + 1:03d}"

run_num = 1
for row in ws.iter_rows(min_row=FIRST_ROW, min_col=5, max_col=6, values_only=True):
    if row[0] == EXP_NAME:
        try: run_num = max(run_num, int(row[1] or 0) + 1)
        except (TypeError, ValueError): pass

now       = datetime.datetime.now()
timestamp = now.strftime("%Y%m%d_%H%M")
print(f"\n🔖  Exp ID: {exp_id}  |  Run #{run_num}  |  {now.strftime('%Y-%m-%d %H:%M')}")


# ── STEP 5: GENERATE & SAVE TEST PREDICTIONS TO DRIVE ─────────────────────────
pred_dir         = os.path.join(DRIVE_PREDS, f"{exp_id}_{EXP_NAME}_{timestamp}")
rest_pred_path   = os.path.join(pred_dir, f"pred_restaurant_{exp_id}.jsonl")
laptop_pred_path = os.path.join(pred_dir, f"pred_laptop_{exp_id}.jsonl")
os.makedirs(pred_dir, exist_ok=True)

def predict_test(test_data, label):
    preds = []
    for item in tqdm(test_data, desc=f"  Predicting {label}"):
        av_list = []
        for aspect in item['Aspect']:
            v, a = predict_va(item['Text'], aspect)
            av_list.append({'Aspect': aspect, 'VA': f'{v:.2f}#{a:.2f}'})
        preds.append({'ID': item['ID'], 'Aspect_VA': av_list})
    return preds

def save_jsonl(preds, path):
    with open(path, 'w') as f:
        for p in preds:
            f.write(json.dumps(p) + '\n')

print(f"\n⏳  Generating test predictions...")
rest_preds   = predict_test(test_rest,   "Restaurant")
laptop_preds = predict_test(test_laptop, "Laptop")
save_jsonl(rest_preds,   rest_pred_path)
save_jsonl(laptop_preds, laptop_pred_path)

# Clean Drive-relative paths for the Excel cell
def drive_rel(p):
    return p.replace("/content/drive/", "")

pred_folder_rel  = drive_rel(pred_dir)
rest_pred_rel    = drive_rel(rest_pred_path)
laptop_pred_rel  = drive_rel(laptop_pred_path)

print(f"\n💾  Predictions saved:")
print(f"    📁 {pred_folder_rel}/")
print(f"       ├── pred_restaurant_{exp_id}.jsonl  ({len(rest_preds)} items)")
print(f"       └── pred_laptop_{exp_id}.jsonl      ({len(laptop_preds)} items)")


# ── STEP 6: BUILD COLUMN STRINGS ──────────────────────────────────────────────
dataset_size_str = (
    f"Train={len(train_pairs)} | "
    f"Test_Rest={n_rest} | Test_Laptop={n_laptop} | "
    f"(full train used, no val split)"
)

custom_metric_str = (
    f"Rest_VAL={rmse_rest_dev} | "
    f"Laptop_VAL={rmse_laptop_dev} | "
    f"Rest_TEST=N/A | Laptop_TEST=N/A"
)

summary_str = (
    f"Runtime: {runtime_str} ({samples_per_s} samp/s) | "
    f"Train loss: {train_loss_str} | "
    f"Val loss: {val_loss_str} | "
    f"BestCkpt: {best_ckpt}"
)

other_hp_str = (
    f"regressor=Linear(768→256→2)+ReLU | dropout=0.1x2 | "
    f"loss=MSE | load_best_model=True | "
    f"eval_strategy={args.eval_strategy}"
)

artifact_str = (
    f"Folder: {pred_folder_rel} || "
    f"Restaurant: {rest_pred_rel} || "
    f"Laptop: {laptop_pred_rel}"
)


# ── STEP 7: WRITE ROW TO EXCEL ────────────────────────────────────────────────
new_row = [
    # IDENTIFICATION (cols 1–6)
    exp_id,
    now.strftime("%Y-%m-%d"),
    now.strftime("%H:%M"),
    RESEARCHER,
    EXP_NAME,
    run_num,
    # EXPERIMENT SETUP (cols 7–12)
    "VA Regression (ABSA)",
    "SemEval Restaurant + Laptop",
    dataset_size_str,
    "PyTorch / HF Transformers",
    MODEL_NAME,                          # roberta-base, pulled from Cell 4
    "Yes",
    # HYPERPARAMETERS (cols 13–20)
    args.learning_rate,
    args.per_device_train_batch_size,
    int(args.num_train_epochs),
    "AdamW",
    "LinearWarmup",
    0.1,
    128,
    other_hp_str,
    # METRICS (cols 21–27)
    None,                                # F1 — N/A for regression task
    None,                                # Precision
    None,                                # Recall
    None,                                # Accuracy
    rmse_overall_test,                   # RMSE (weighted combined test)
    None,                                # BLEU / ROUGE
    custom_metric_str,                   # Custom → per-domain RMSE breakdown
    # NOTES (cols 28–32)
    "✅ Success",
    summary_str,                         # Training dynamics + runtime
    "None",
    NEXT_STEPS,
    artifact_str,                        # Drive paths to prediction files
]

# Find next empty row
next_row = FIRST_ROW
for row in ws.iter_rows(min_row=FIRST_ROW, max_col=1, values_only=True):
    if row[0]: next_row += 1
    else: break

for col_idx, val in enumerate(new_row, 1):
    ws.cell(row=next_row, column=col_idx, value=val)

# Make the Linked Artifact cell (col 32) a hyperlink to a Drive search for the exp ID
artifact_cell           = ws.cell(row=next_row, column=32)
artifact_cell.hyperlink = f"https://drive.google.com/drive/search?q={exp_id}"
artifact_cell.font      = Font(name="Arial", size=10, color="7C3AED", underline="single")

wb.save(EXCEL_PATH)


# ── STEP 8: FINAL SUMMARY ─────────────────────────────────────────────────────
print(f"\n{'═'*64}")
print(f"  ✅  LOGGED SUCCESSFULLY TO EXCEL")
print(f"{'═'*64}")
print(f"  Exp ID           : {exp_id}")
print(f"  Project          : {EXP_NAME}")
print(f"  Run #            : {run_num}")
print(f"  Training time    : {runtime_str}")
print(f"  Final train loss : {final_train_loss}")
print(f"  Final val loss   : {final_val_loss}")
print(f"  Best checkpoint  : {best_ckpt}")
print(f"  RMSE Overall TEST: {rmse_overall_test}")
print(f"  RMSE Restaurant  : {rmse_rest_test}")
print(f"  RMSE Laptop      : {rmse_laptop_test}")
print(f"  Predictions at   : {pred_folder_rel}")
print(f"  Excel row        : {next_row}  ({EXCEL_PATH})")
print(f"{'═'*64}")